# Замороженный MPNet и CatBoost

Проверяем, помогают ли готовые смысловые представления отзывов предсказывать оценку 1–5. Энкодер sentence-transformers/all-mpnet-base-v2 не дообучается: он один раз превращает текст в 768 чисел. CatBoostRegressor обучается на этих векторах с функцией потерь MAE. Разбиение совпадает с modeling.ipynb: валидационные тексты не пересекаются с обучающими.

In [1]:
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import gc
import os

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostRegressor
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedGroupKFold

root = Path.cwd()
if not (root / "data" / "raw" / "train.csv").exists():
    root = root.parent
data_path = root / "data" / "raw" / "train.csv"
if not data_path.exists():
    raise FileNotFoundError("Поместите train.csv в data/raw/")
train = pd.read_csv(data_path)
assert list(train.columns) == ["Id", "Review", "Rating"]
clean = train.drop_duplicates(subset=["Review"], keep="first").reset_index(drop=True)
groups = clean["Review"].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026)
train_idx, valid_idx = next(splitter.split(clean["Review"], clean["Rating"], groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[valid_idx]))
y_train = clean["Rating"].iloc[train_idx].to_numpy()
y_valid = clean["Rating"].iloc[valid_idx].to_numpy()
print(f"Обучение: {len(train_idx):,}; валидация: {len(valid_idx):,}")

/Users/vlanuka/Documents/ChatGPT/ncot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Обучение: 47,980; валидация: 11,996


## Векторизация отзывов

Используем закреплённую версию MPNet. На компьютере с 8 ГБ памяти ограничиваем вход 256 токенами и кодируем небольшими пакетами. Векторы сохраняются в локальном data/cache/, который исключён из Git: повторный запуск не будет пересчитывать их. Кэш зависит от содержимого train.csv, версии MPNet и длины входа.

In [2]:
model_id = "sentence-transformers/all-mpnet-base-v2"
model_revision = "e8c3b32edf5434bc2275fc9bab85f82640a19130"
max_length = 256
device = "mps" if torch.backends.mps.is_available() else "cpu"
fingerprint = sha256(data_path.read_bytes()).hexdigest()[:12]
cache_dir = root / "data" / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
cache_path = cache_dir / f"mpnet_v1_{model_revision[:8]}_{max_length}_{fingerprint}.npy"
expected_shape = (len(clean), 768)

if not cache_path.exists():
    started = perf_counter()
    encoder = SentenceTransformer(model_id, revision=model_revision, device=device)
    encoder.max_seq_length = max_length
    partial_path = cache_path.with_suffix(".partial.npy")
    vectors = np.lib.format.open_memmap(
        partial_path, mode="w+", dtype=np.float32, shape=expected_shape
    )
    chunk_size = 256
    for start in range(0, len(clean), chunk_size):
        stop = min(start + chunk_size, len(clean))
        vectors[start:stop] = encoder.encode(
            clean["Review"].iloc[start:stop].tolist(),
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        if device == "mps":
            torch.mps.empty_cache()
        if stop % 5120 < chunk_size or stop == len(clean):
            vectors.flush()
            print(f"Закодировано {stop:,} / {len(clean):,}", flush=True)
    vectors.flush()
    del vectors, encoder
    gc.collect()
    if device == "mps":
        torch.mps.empty_cache()
    os.replace(partial_path, cache_path)
    print(f"Время кодирования: {(perf_counter() - started) / 60:.1f} мин")
else:
    print("Использован локальный кэш эмбеддингов")
embeddings = np.load(cache_path, mmap_mode="r")
assert embeddings.shape == expected_shape
assert embeddings.dtype == np.float32
print("Размер матрицы:", embeddings.shape, "устройство:", device)

Использован локальный кэш эмбеддингов
Размер матрицы: (59976, 768) устройство: mps


## Обучение регрессора

CatBoost получает только эмбеддинги обучающих отзывов и их оценки. Количество деревьев и остальные параметры зафиксированы до оценки на валидации. Выход ограничиваем допустимым диапазоном 1–5; округление не требуется, поскольку метрика Kaggle принимает дробные предсказания.

In [3]:
X_train = np.asarray(embeddings[train_idx], dtype=np.float32)
X_valid = np.asarray(embeddings[valid_idx], dtype=np.float32)
assert X_train.shape == (len(train_idx), 768)
assert X_valid.shape == (len(valid_idx), 768)

regressor = CatBoostRegressor(
    loss_function="MAE",
    iterations=500,
    depth=6,
    learning_rate=0.05,
    thread_count=4,
    random_seed=2026,
    verbose=False,
    allow_writing_files=False,
)
regressor_path = cache_dir / f"catboost_mpnet_v1_{model_revision[:8]}_{max_length}_{fingerprint}_split2026.cbm"
if regressor_path.exists():
    regressor.load_model(str(regressor_path))
    print("Использован локальный кэш CatBoost")
else:
    started = perf_counter()
    regressor.fit(X_train, y_train)
    regressor.save_model(str(regressor_path))
    print(f"Обучение CatBoost: {(perf_counter() - started) / 60:.1f} мин")
predictions = np.clip(regressor.predict(X_valid), 1, 5)
mae = mean_absolute_error(y_valid, predictions)
print(f"MAE MPNet + CatBoost: {mae:.5f}")

by_rating = pd.DataFrame({
    "строк": [int((y_valid == rating).sum()) for rating in range(1, 6)],
    "MAE": [
        mean_absolute_error(y_valid[y_valid == rating], predictions[y_valid == rating])
        for rating in range(1, 6)
    ],
}, index=pd.Index(range(1, 6), name="Rating"))
display(by_rating.round(4))

Использован локальный кэш CatBoost
MAE MPNet + CatBoost: 0.44325


,строк,MAE
Rating,,
1,3732,0.5765
2,326,1.0141
3,336,1.1681
4,670,0.8963
5,6932,0.2657


## Выводы

- MPNet + CatBoost дал **MAE 0,44325** на той же отложенной части, где TF-IDF + Logistic Regression дал **0,19165**. Разница — **+0,25160 MAE**: в этом эксперименте готовые MPNet-векторы оказались заметно слабее простого текстового бейзлайна.
- По оценкам 1 и 5 MAE составила 0,5765 и 0,2657 соответственно. По редким оценкам 2–4 модель также часто ошибается; общий результат нельзя объяснить только их малой численностью.
- Получение векторов заняло 16,6 мин на Apple M1; обучение CatBoost — 0,5 мин. Это измерения на данном компьютере, а не универсальные оценки скорости.
- Энкодер не адаптировался к задаче рейтинга. Возможное объяснение слабого результата — общая смысловая близость отзывов не всегда отражает разницу между оценками. Проверим эту гипотезу дообучением DeBERTa; преимущество не предполагаем заранее.